# Whale and Boat Detection: Conditional DETR

Finds each whale and boat in a 500x500 SkySat chip and reports where it is, rather than giving
the whole chip one label. Two detectors are trained the same way on the same split:
`train_faster_rcnn.ipynb` (CNN baseline) and `train_detr.ipynb` (DETR, the focus).
`evaluate_detectors.ipynb` compares them on the test split.

**Model.** Conditional DETR (Meng et al., 2021), ResNet-50 backbone, pretrained on COCO
(`microsoft/conditional-detr-resnet-50`). DETR has no anchors and no NMS: a transformer decoder
turns 300 learned *object queries* into 300 predictions, and training matches predictions to
labels one-to-one (Hungarian matching). Each query looks at the image through cross-attention,
which `evaluate_detectors.ipynb` visualizes. Plain DETR needs ~500 epochs to converge; the
conditional variant converges several times faster, which matters with ~1,400 training chips.
If it still won't learn small whales, set `DETR_CHECKPOINT = 'SenseTime/deformable-detr'`
(Deformable DETR, multi-scale features) and rerun; nothing else changes.

**Labels are points.** Annotators clicked each whale and boat once. For training, each point
becomes a square box of a fixed size per class (a modelling choice, so it is tuned). For scoring,
a detection is correct if its center is within `MATCH_RADIUS_M` of a point of the same class,
so the invented box size never affects the score. Whales marked "possibly" (certainty 0) are not
training targets and don't count either way when scoring.

**Missing a whale is the costly error.** The researchers would rather check a false alarm than
lose a whale. So models and settings are compared by **average precision (AP)**, which covers
every score cutoff. After training, the cutoff is set on validation to find `TARGET_RECALL` of
the whales and boats, and we report what that costs in false alarms.

**Outputs** go to `checkpoints/<RUN_NAME>/`: `run.json` (data, split, settings, search results,
cutoffs, history), `detector.pt` (weights) and `optuna.db` (the search, resumable). Only
validation scores are reported here; the test split is used once, in `evaluate_detectors.ipynb`.

## 1. Setup

In [ ]:
import os
import sys

os.environ.setdefault('CUDA_MODULE_LOADING', 'LAZY')
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q rasterio timm "transformers>=5.20" optuna

In [ ]:
import gc
import json
import random
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import optuna
import pandas as pd
import torch

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists()),
    Path.cwd(),
)
sys.path.insert(0, str(ROOT / 'src'))
from whaleboat.data import band_stretch, describe_split, make_split  # noqa: E402
from whaleboat.detection import (  # noqa: E402
    DET_CLASSES,
    PIXEL_M,
    ChipCache,
    DetectionDataset,
    count_points,
    evaluate_detections,
    load_detection_chips,
    make_det_loader,
    pr_curve,
    summarize,
    threshold_for_recall,
    weather_aug,
)
from whaleboat.detectors import build_detector, fit, predict_loader, seed_everything  # noqa: E402

print(f'PyTorch {torch.__version__}, CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU {props.name}, {props.total_memory / 1e9:.1f} GB')
else:
    print('WARNING: no GPU. Use a GPU runtime (Colab: Runtime > Change runtime type).')

## 2. Config

Sized for a Colab T4 (16 GB GPU, ~12 GB RAM). On a smaller GPU, lower `BATCH_SIZE`; the batch
is also shrunk automatically when a trial upsamples the chips (memory grows with image area).

- **Main run:** `RUN_NAME = 'detr_main'`, `RUN_SEARCH = True`.
- **Weather ablation:** `RUN_NAME = 'detr_noweather'`, `WEATHER_AUG = False`,
  `SPLIT_FROM = PARAMS_FROM = 'detr_main'`. It reuses the main run's split and tuned settings,
  so weather augmentation is the only difference.
- **On Colab,** `CKPT_DIR` points at Google Drive so runs survive a disconnect. The search is
  stored in `optuna.db` and continues where it stopped when rerun with the same `RUN_NAME`.

In [ ]:
MODEL = 'detr'
DETR_CHECKPOINT = 'microsoft/conditional-detr-resnet-50'  # or 'SenseTime/deformable-detr'
RUN_NAME = 'detr_main'
DATASETS = ['original', 'skysat']
SPLIT_MODE = 'group'  # 'group' (default), 'date' (hold out whole days) or 'random'
SPLIT_FROM = None  # a run name: reuse its exact split
PARAMS_FROM = None  # a run name: reuse its tuned settings and skip the search
WEATHER_AUG = True  # the ablation switch

TARGET_RECALL = 0.90  # share of whales (and boats) the operating cutoff must find on validation
MATCH_RADIUS_M = 10.0  # a detection within this distance of a point finds it
SELECTION_METRIC = 'mean_ap'  # validation score that picks settings: 'mean_ap' or 'whale_ap'

RUN_SEARCH = True
N_TRIALS = 15
TUNE_EPOCHS = 15  # short runs for the search; weak trials are stopped early
FINAL_EPOCHS = 60  # the chosen settings, trained properly
PATIENCE = 12
BATCH_SIZE = 8  # at scale 1; divided by scale^2 when chips are upsampled
GRAD_CLIP = 0.1
USE_AMP = True  # mixed precision: faster on T4/A100, less memory
NUM_WORKERS = 2
SEED = 42

DATA_DIR = ROOT / 'data'
ORIG_DIR = DATA_DIR / 'whale' / 'data'
SKYSAT_DIR = DATA_DIR / 'skysat_chips_500'
CKPT_DIR = ROOT / 'checkpoints'
if IN_COLAB:
    from google.colab import drive

    drive.mount('/content/drive')
    CKPT_DIR = Path('/content/drive/MyDrive/whaleboat/checkpoints')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RUN_DIR = CKPT_DIR / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
RADIUS_PX = MATCH_RADIUS_M / PIXEL_M
seed_everything(SEED)
print(f'{MODEL} -> {RUN_DIR}  weather_aug={WEATHER_AUG}  device={DEVICE}')

## 3. Chips, points and split

The split is the same scene-grouped split the classifier used: chips from the same day whose
footprints overlap stay in one part, so the same whale or boat is never in train and test.

In [ ]:
chips = load_detection_chips(DATASETS, ORIG_DIR, SKYSAT_DIR)
by_name = {c['name']: c for c in chips}
if SPLIT_FROM:
    src_split = json.loads((CKPT_DIR / SPLIT_FROM / 'run.json').read_text())['split']
    split = {k: [n for n in v if n in by_name] for k, v in src_split.items()}
else:
    split = make_split(chips, mode=SPLIT_MODE, seed=SEED)
parts = {k: [by_name[n] for n in v] for k, v in split.items()}
describe_split(chips, split)
for k, cs in parts.items():
    print(f'{k:5s} objects: {count_points(cs)}')

## 4. Load chips into memory

Each chip is stretched to 8-bit RGB with one fixed per-band stretch taken from training chips,
then kept in RAM (~1 GB for train + val) so the many short search runs don't wait on disk.

In [ ]:
band_lo, band_hi = band_stretch([c['path'] for c in parts['train']], seed=SEED)
cache = ChipCache(parts['train'] + parts['val'], band_lo, band_hi)
print(f'{len(cache)} chips cached, {sum(v.nbytes for v in cache.values()) / 1e9:.2f} GB')

### What the model sees

Training boxes (made from points) on a few chips, then one chip under several random weather
augmentations: haze (uniform or patchy), sun glint, exposure, color cast, noise and blur.

In [ ]:
show = DetectionDataset(
    random.Random(1).sample([c for c in parts['train'] if c['points']], 4),
    cache,
    box_px={'whale': 32, 'boat': 32},
)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (img, t) in zip(axes, show):
    ax.imshow(img.permute(1, 2, 0))
    for (x0, y0, x1, y1), lab in zip(t['boxes'].tolist(), t['labels'].tolist()):
        color = 'red' if DET_CLASSES[lab] == 'whale' else 'yellow'
        ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, color=color))
    ax.set_title(t['name'].split('/')[-1][:28], fontsize=8)
    ax.axis('off')
plt.show()

img, _ = show[0]
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
axes[0].imshow(img.permute(1, 2, 0))
axes[0].set_title('original')
for ax in axes[1:]:
    ax.imshow(weather_aug(img.clone(), p=0.6).permute(1, 2, 0))
    ax.set_title('weather augmentation')
for ax in axes:
    ax.axis('off')
plt.show()

## 5. Hyperparameter search

**Tuned** (Optuna's TPE sampler, `N_TRIALS` runs of `TUNE_EPOCHS` epochs; a median pruner stops
trials that fall behind the others, so most of the budget goes to promising settings):

| Setting | Range | Why it matters here |
| --- | --- | --- |
| Input scale | x1, x1.6, x2 (chips upsampled) | Whales are ~20-40 px; detectors, DETR above all, miss objects that small |
| Box size from each point | 24, 32 or 48 px, whale and boat separately | We invent the boxes, so their size is a modelling choice; scoring ignores it |
| Learning rate | 3e-5 to 3e-4 (log) | The most important optimizer setting for fine-tuning |
| Weight decay | 1e-5 to 1e-2 (log) | ~1,400 training chips: overfitting is a real risk |
| Water chips per epoch | as in the data (~55%) or 30% | More water teaches what not to flag; less gives the model more whales to learn from |

**Fixed** (stated so the report can list them):

| Setting | Value | Why not tuned |
| --- | --- | --- |
| Optimizer and schedule | AdamW, 5% warmup, cosine decay | Standard for fine-tuning both models |
| Batch size | `BATCH_SIZE` / scale^2 | Set by GPU memory, not a modelling choice |
| Flips and 90-degree rotations | always on | Overhead imagery has no "up" |
| Weather augmentation | `WEATHER_AUG` | It is the ablation, so it is held fixed during the search |
| Match radius | `MATCH_RADIUS_M` | Defines what counts as found; tuning it would game the metric |
| Score cutoff | set afterwards on validation | An operating choice, made for `TARGET_RECALL` |
| Backbone learning rate | 0.1 x the main rate | The DETR recipe; protects the pretrained CNN features |
| Loss weights and matching costs | class 2, L1 box 5, GIoU 2 (library defaults) | Well studied; expensive to search |
| Auxiliary decoder losses | on | As in the DETR papers; speeds up convergence |
| Object queries | 300 | Pretrained queries; changing the count discards them |
| Gradient clipping | norm 0.1 | The DETR recipe; training is unstable without it |
| Class-head bias | 1% object prior | Standard focal-loss start; a fresh 0 bias swamps the box losses |

Every trial is scored on the **validation** split by `SELECTION_METRIC` (mean of whale and boat
AP by default). The test split is not touched.

In [ ]:
def suggest(trial):
    p = {
        'scale': trial.suggest_categorical('scale', [1.0, 1.6, 2.0]),
        'whale_box': trial.suggest_categorical('whale_box', [24, 32, 48]),
        'boat_box': trial.suggest_categorical('boat_box', [24, 32, 48]),
        'lr': trial.suggest_float('lr', 3e-5, 3e-4, log=True),
        'weight_decay': trial.suggest_float('weight_decay', 1e-5, 1e-2, log=True),
        'water': trial.suggest_categorical('water', ['as_is', 'less']),
    }
    return p


def loaders(p):
    box = {'whale': p['whale_box'], 'boat': p['boat_box']}
    bs = max(1, round(BATCH_SIZE / p['scale'] ** 2))
    water = None if p['water'] == 'as_is' else 0.3
    train = make_det_loader(
        parts['train'],
        cache,
        bs,
        p['scale'],
        box,
        train=True,
        weather=WEATHER_AUG,
        water_frac=water,
        workers=NUM_WORKERS,
        seed=SEED,
    )
    val = make_det_loader(parts['val'], cache, bs * 2, p['scale'], box, workers=NUM_WORKERS)
    return train, val


def make_evaluate(val_loader, scale):
    def evaluate(det):
        preds = predict_loader(det, val_loader, scale, DEVICE, amp=USE_AMP)
        m = summarize(evaluate_detections(preds, parts['val'], RADIUS_PX))
        return {'score': m[SELECTION_METRIC], 'whale_ap': m['whale_ap'], 'boat_ap': m['boat_ap']}

    return evaluate


def train(p, epochs, patience=None, on_epoch=None, log=print):
    size = round(500 * p['scale'])
    det = build_detector(MODEL, size, p, detr_checkpoint=DETR_CHECKPOINT)
    train_loader, val_loader = loaders(p)
    history = fit(
        det,
        train_loader,
        epochs,
        p['lr'],
        p['weight_decay'],
        DEVICE,
        evaluate=make_evaluate(val_loader, p['scale']),
        clip=GRAD_CLIP,
        amp=USE_AMP,
        patience=patience,
        on_epoch=on_epoch,
        log=log,
    )
    return det, history, val_loader


def free(*objs):
    for o in objs:
        del o
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
def objective(trial):
    p = suggest(trial)
    seed_everything(SEED + trial.number)

    def report(epoch, score):
        trial.report(score, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    print(f'trial {trial.number}: {p}')
    det = None
    try:
        det, history, _ = train(p, TUNE_EPOCHS, on_epoch=report, log=None)
        best = max(h['val_score'] for h in history)
        trial.set_user_attr('whale_ap', max(h['val_whale_ap'] for h in history))
        trial.set_user_attr('minutes', sum(h['seconds'] for h in history) / 60)
        print(f'  -> {SELECTION_METRIC} {best:.4f}')
        return best
    except torch.OutOfMemoryError:
        print('  -> out of GPU memory; lower BATCH_SIZE')
        raise optuna.TrialPruned()
    finally:
        free(det)


study = None
if PARAMS_FROM:
    prev = json.loads((CKPT_DIR / PARAMS_FROM / 'run.json').read_text())
    best_params = prev['params']
    print(f'Using the settings tuned in {PARAMS_FROM}: {best_params}')
else:
    study = optuna.create_study(
        study_name=RUN_NAME,
        storage=f'sqlite:///{RUN_DIR / "optuna.db"}',
        load_if_exists=True,
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=SEED),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=4, n_warmup_steps=3),
    )
    done = len([t for t in study.trials if t.state.is_finished()])
    if RUN_SEARCH and done < N_TRIALS:
        optuna.logging.set_verbosity(optuna.logging.WARNING)
        study.optimize(objective, n_trials=N_TRIALS - done)
    best_params = study.best_params
    print(f'Best {SELECTION_METRIC} {study.best_value:.4f} with {best_params}')

In [ ]:
if study is not None:
    trials = study.trials_dataframe(attrs=('number', 'value', 'params', 'state', 'user_attrs'))
    trials.columns = [c.replace('params_', '').replace('user_attrs_', '') for c in trials.columns]
    display(trials.sort_values('value', ascending=False))
    trials.to_csv(RUN_DIR / 'search_trials.csv', index=False)

    fig, ax = plt.subplots(figsize=(7, 3.5))
    done = trials[trials.state == 'COMPLETE']
    ax.scatter(done.number, done.value, label='finished')
    pruned = trials[trials.state == 'PRUNED']
    ax.scatter(pruned.number, pruned.value, marker='x', color='gray', label='stopped early')
    ax.plot(done.number, done.value.cummax(), color='black', lw=1, label='best so far')
    ax.set(xlabel='trial', ylabel=f'validation {SELECTION_METRIC}', title='Search progress')
    ax.legend()
    plt.show()
    try:  # which settings mattered (fANOVA over finished trials)
        imp = optuna.importance.get_param_importances(study)
        print('Setting importance:', {k: round(v, 2) for k, v in imp.items()})
    except (RuntimeError, ValueError) as e:
        print(f'Importance not available yet: {e}')

## 6. Train the chosen settings

Full length, keeping the epoch with the best validation score (early stopping after `PATIENCE`
epochs without improvement).

In [ ]:
seed_everything(SEED)
det, history, val_loader = train(best_params, FINAL_EPOCHS, patience=PATIENCE)
torch.save(
    {'model_state_dict': det.state_dict(), 'params': best_params, 'model': MODEL},
    RUN_DIR / 'detector.pt',
)

h = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot(h.epoch, h.loss)
axes[0].set(xlabel='epoch', ylabel='training loss', title='Training loss')
for k in ['whale_ap', 'boat_ap']:
    axes[1].plot(h.epoch, h[f'val_{k}'], label=k.replace('_', ' '))
axes[1].set(xlabel='epoch', ylabel='validation AP', title='Validation AP', ylim=(0, 1))
axes[1].legend()
plt.show()

## 7. Validation results and the operating cutoff

The cutoff for each class is the highest score that still finds `TARGET_RECALL` of that class's
points on validation. `evaluate_detectors.ipynb` applies these same cutoffs to the test split.

In [ ]:
val_preds = predict_loader(det, val_loader, best_params['scale'], DEVICE, amp=USE_AMP)
val_res = evaluate_detections(val_preds, parts['val'], RADIUS_PX)
thresholds = {k: threshold_for_recall(val_res[k], TARGET_RECALL) for k in DET_CLASSES}
val_metrics = summarize(val_res, thresholds)

rows = []
for k in DET_CLASSES:
    rows.append(
        {
            'class': k,
            'objects': val_metrics[f'{k}_n'],
            'AP': val_metrics[f'{k}_ap'],
            'cutoff': thresholds[k],
            'recall': val_metrics[f'{k}_recall'],
            'precision': val_metrics[f'{k}_precision'],
            'false alarms / km2': val_metrics[f'{k}_fa_per_km2'],
        }
    )
print(
    f'Validation, {len(parts["val"])} chips, match radius {MATCH_RADIUS_M:g} m, '
    f'cutoff for {TARGET_RECALL:.0%} recall:'
)
display(pd.DataFrame(rows).round(3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, k in zip(axes, DET_CLASSES):
    prec, rec, _ = pr_curve(val_res[k])
    ax.plot(rec, prec)
    ax.scatter(
        [val_metrics[f'{k}_recall']],
        [val_metrics[f'{k}_precision']],
        color='red',
        zorder=3,
        label=f'cutoff {thresholds[k]:.2f}',
    )
    ax.set(
        xlabel='recall',
        ylabel='precision',
        xlim=(0, 1),
        ylim=(0, 1.02),
        title=f'{k}: AP {val_metrics[f"{k}_ap"]:.3f} (val, n={val_metrics[f"{k}_n"]})',
    )
    ax.legend()
plt.show()

In [ ]:
run = {
    'run_name': RUN_NAME,
    'model': MODEL,
    'detr_checkpoint': DETR_CHECKPOINT if MODEL == 'detr' else None,
    'created': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'datasets': DATASETS,
    'split_mode': SPLIT_MODE,
    'split_from': SPLIT_FROM,
    'split': split,
    'band_lo': band_lo,
    'band_hi': band_hi,
    'seed': SEED,
    'weather_aug': WEATHER_AUG,
    'params': best_params,
    'params_from': PARAMS_FROM,
    'fixed': {
        'tune_epochs': TUNE_EPOCHS,
        'final_epochs': FINAL_EPOCHS,
        'patience': PATIENCE,
        'batch_size_at_scale_1': BATCH_SIZE,
        'grad_clip': GRAD_CLIP,
        'amp': USE_AMP,
        'n_trials': N_TRIALS,
        'selection_metric': SELECTION_METRIC,
    },
    'match_radius_m': MATCH_RADIUS_M,
    'target_recall': TARGET_RECALL,
    'thresholds': thresholds,
    'val_metrics': val_metrics,
    'history': history,
    'versions': {'torch': torch.__version__, 'optuna': optuna.__version__},
}
(RUN_DIR / 'run.json').write_text(json.dumps(run, indent=1, default=float))
print(f'Saved {RUN_DIR / "run.json"} and {RUN_DIR / "detector.pt"}')